# 수정 필요 
- 품목명 중복
- MTI 코드

# 데이터 불러오기

In [1]:
import pandas as pd  

In [128]:
hs_import = pd.read_csv('./datasets/수출입 금액_건수.csv')
hs_import.head(2)
hs_import.info()
hs_import['HS_CD'] = hs_import['HS_CD'].astype(str) 
hs_import.sort_values(by="HS_CD").head()
# hs_import['MTI_CD'] = hs_import['MTI_CD'].astype(str) 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10720 entries, 0 to 10719
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   HS_CD    10720 non-null  int64
 1   EXP_AMT  10720 non-null  int64
 2   IMP_AMT  10720 non-null  int64
 3   count    10720 non-null  int64
dtypes: int64(4)
memory usage: 335.1 KB


,HS_CD,EXP_AMT,IMP_AMT,count
4481,1001190000,171,40854768,1791
10088,1001919000,72,801,60
4149,1001991090,0,1473696267,2961
9442,1001992010,0,81,36
2532,1001992090,0,2649109299,798


In [109]:
hs_ems = pd.read_csv('./datasets/ems_hscode.csv')
hs_ems.head(2)

hs_ems = hs_ems.drop_duplicates()    # 중복값 거거
hs_ems[hs_ems['KOR_NAME'] == '립밤']

hs_ems['HS_CD'] = hs_ems['HS_CD'].astype(str) 

# hs_33_ems = hs_ems[(hs_ems['HS_CD'] >= 3300000000) & (hs_ems['HS_CD'] < 3400000000)].sort_values('HS_CD')
# hs_33_ems.head(2)

# 중복 값이 꽤 많다. 
# hs_33_ems[hs_33_ems.duplicated()]
# df.drop_duplicates()
# len(hs_33_ems)


In [110]:
hs_items = pd.read_csv('./datasets/HS_CODE.csv',encoding='cp949') 
hs_items['HS_CD'] = hs_items['HS_CD'].astype(str) 
hs_items.info()

# hs_33_items = hs_items[hs_items['HS_CD'].astype(str).str.match(r'^33')]           
# hs_33_items.head(10)                                                           
# hs_items[hs_items['HS_CD'] > 3300000000]                                    
# hs_33_items.info()                                 
# len(hs_33_items)                                

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31120 entries, 0 to 31119
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   HS_CD     31120 non-null  object
 1   KOR_NAME  31120 non-null  object
 2   ENG_NAME  31120 non-null  object
dtypes: object(3)
memory usage: 729.5+ KB


In [111]:
# hs_33_items[hs_33_items['HS_CD'] == 3307909000]
# hs_33_ems[hs_33_ems['HS_CD'] == 3307909000]

In [13]:
hs_mti = pd.read_csv('./datasets/KTS_CODE_CONVERSION.csv', sep='|', encoding='utf-8')
# hs_mti_33 = hs_mti[hs_mti['HS_CD'].astype(str).str.match(r'^33')]
hs_mti['MTI_CD'] = hs_mti['MTI_CD'].astype(str) 
hs_mti['YEAR'] = hs_mti['YEAR'].astype(str) 
hs_mti['HS_CD'] = hs_mti['HS_CD'].astype(str) 

hs_mti.head(5) 
# MTI_CD는 HS_CD 10자리 값의 쌍으로만 재재
hs_mti[hs_mti['MTI_CD'] == '827220']

,YEAR,HS_CD,MTI_CD


In [119]:
# MTI CODE 품목명 데이터 불러오기  
mti = pd.read_csv('./datasets/MTI_CODE.csv',encoding='cp949')
mti['MTI_CD'] = mti['MTI_CD'].astype(str)
# mti['HS_CD'] = mti['HS_CD'].astype(str) 

mti.head(2)

,MTI_CD,KOR_NAME,ENG_NAME
0,0,농림수산물,agricultural & forest & marineproducts
1,1,광산물,mineralproduct


In [ ]:
# 엑셀에서 utf-8로 다시 저장 한 경우 encoding='utf-8' 적용 가능 ! 
# MTI_CODE2 = pd.read_csv('./datasets/MTI_CODE_UTF8.csv',encoding='utf-8')
# MTI_CODE2.head(2)

# 사용자 함수 

In [114]:
def remove_substrings(names):
    # 리스트 내의 문자열 중, 다른 문자열에 포함된 경우 해당 문자열을 제거합니다.
    # 결과를 담을 리스트 초기화
    filtered = []
    for name in names:
        # name이 자기 자신이 아닌 다른 항목에 포함되어 있다면 건너뜁니다.
        if any(name != other and name in other for other in names):
            continue
        filtered.append(name)
    return filtered

# 데이터 합치기

In [120]:
# 우체국 품목명 병합!
# pd.merge(df1, df2, how='outer') (합집합)                 
 
# 두 데이터프레임에서 HS_CD와 KOR_NAME 컬럼만 추출하고, 하나로 합침
combined = pd.merge(hs_ems, hs_items, how='outer')
# display(combined.head(2))

# HS_CD를 기준으로 그룹화하여 KOR_NAME을 쉼표(,)로 연결                                                    
hs = combined.groupby("HS_CD")[["KOR_NAME","ENG_NAME"]].agg(lambda x: ", ".join(remove_substrings(x))).reset_index()

# display(result.head(2))                            
# hs[hs['HS_CD'] > 3300000000].head(5)    # str변환해서 오류      

In [121]:
# HS_KNAME_ENAME_YEAR_MTI

hs = pd.merge(hs, hs_mti, on='HS_CD', how='oute
r')                                  
hs[hs['MTI_CD'].notna()].head()  # MTI_CD 값이 있는 부분만 들고오고 싶을 때  
# hs.info()
# combined.info()    
# hs[hs['MTI_CD'].isna()].head()  # HS_CD 10자리인 값만 MTI_CD가 존재한다!!!

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31083 entries, 0 to 31082
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   HS_CD     31083 non-null  object
 1   KOR_NAME  31083 non-null  object
 2   ENG_NAME  31083 non-null  object
 3   YEAR      11886 non-null  object
 4   MTI_CD    11886 non-null  object
dtypes: object(5)
memory usage: 1.2+ MB


In [116]:
# HS_CD STRING 변환 후 오류 
# hs_33 = hs[(hs['HS_CD']>=3300000000) & (hs['HS_CD']<3400000000)]
hs_33[hs_33['MTI_CD'].notna()]
hs_33.info()

TypeError: '>=' not supported between instances of 'str' and 'int'

In [117]:
# hs[hs['HS_CD'] == 3304301000]

### HS_K품목_E품목_MTI_MTI품목

In [125]:
# HS_KNAME_ENAME_YEAR_MTI_MTINAME

hs = pd.merge(hs, mti, on='MTI_CD', how='outer')
hs[(hs['MTI_CD'].notna()) & (hs['HS_CD'].notna()) ].head(2)
# hs1[(hs1['MTI_CD'].notna()) & (hs1['HS_CD'] >= 3300000000) & (hs1['HS_CD']<3400000000)].head().  # MTI_CD 값이 있는 부분만 들고오고 싶을 때         
# combined.info()                                                                                 

,HS_CD,KOR_NAME_x,ENG_NAME_x,YEAR,MTI_CD,KOR_NAME_y,ENG_NAME_y
8,1006100000,벼,Rice i the husk (paddy o ough),2024,11110,쌀,rice
9,1006201000,메현미,Noglutious,2024,11110,쌀,rice


### HS_K품목_E품목_MTI_MTINAME + 수출_수입_건수

In [132]:
hs = pd.merge(hs, hs_import, on='HS_CD', how='outer')                                  
hs[hs['MTI_CD'].notna() & (hs['HS_CD'].notna())].head()      
## 수출, 수입 금액이 없는 값 존재.. -> 원본데이터에도 없음 

,HS_CD,KOR_NAME_x,ENG_NAME_x,YEAR,MTI_CD,KOR_NAME_y,ENG_NAME_y,EXP_AMT,IMP_AMT,count
7,1001110000,종자,Seed,2024,11130,밀,wheat,NaN,NaN,NaN
9,1001190000,기타,Othe,2024,11130,밀,wheat,171.0,40854768.0,1791.0
20,1001911000,메슬린(mesli),Mesli,2024,11130,밀,wheat,NaN,NaN,NaN
21,1001919000,기타,Othe,2024,11130,밀,wheat,72.0,801.0,60.0
25,1001991010,메슬린(mesli),Mesli,2024,13600,사료,forage,NaN,NaN,NaN


### 중복 품목명 제거  

# 저장

In [133]:
hs.to_csv("HS_KP_EP_YEAR_MTI_MTIPS_EXP_IMP_COUNT.csv", index=False, encoding="utf-8")